# Medical-VQA — Notebook TUNED (bản CLEAN)

Flow gọn: **Setup → CONFIG → Train (A1, A2, P, D-RL) → EVAL ĐẦY ĐỦ (2-in-1) → Report**.

### Tinh chỉnh (chỉ qua param, KHÔNG sửa `.py`)
- **LR tách theo model:** GPT-2 `LR_GPT2 = 1e-4` (thay 1e-3) — tránh quên kiến thức pretrained; model nhẹ giữ `1e-3`.
- **Weight decay GPT-2:** `WD_GPT2 = 1e-2` (thay 1e-4) — chống overfit ~1600 mẫu.
- Repo/Project hậu tố `-tuned` để không đè baseline.

### Eval
- Một cell **EVAL ĐẦY ĐỦ** (BƯỚC 10): mỗi model load 1 lần → test **cả VQA-RAD + RadImageNet** → push HF (RadImageNet lồng `<run>/radimagenet/`).

### Ngoài phạm vi param (cần sửa code)
- Label smoothing, LR scheduler, discriminative LR, semantic reward — xem `giai_phap_tinh_chinh_kien_truc.html`.

> Khôi phục checkpoint/vocab khi **mất session** đã dời xuống **Phụ lục** cuối notebook (chỉ chạy khi cần).

## BƯỚC 1: CHUẨN BỊ MÔI TRƯỜNG

In [ ]:
# 1.1. Cap nhat ma nguon moi nhat tu GitHub. DAN token vao GITHUB_TOKEN (khong commit token).
GITHUB_TOKEN = ''   # <-- DAN TOKEN O DAY
GITHUB_USER  = 'HoangVuSnape'
REPO = '02_Code_DeepLearning'

import os
CLONE_URL = 'https://' + GITHUB_USER + ':' + GITHUB_TOKEN + '@github.com/' + GITHUB_USER + '/' + REPO + '.git'
if os.path.isdir(REPO + '/.git'):
    print('Repo da ton tai -> git reset --hard origin/main...')
    get_ipython().system('cd ' + REPO + ' && git fetch origin && git reset --hard origin/main && git log --oneline -1')
elif not os.path.isdir('.git'):
    get_ipython().system('git clone ' + CLONE_URL)

In [ ]:
# 1.2. Vao thu muc du an
import os
if os.path.isdir('02_Code_DeepLearning'):
    os.chdir('02_Code_DeepLearning')
print('CWD:', os.getcwd())

In [ ]:
# 1.3. Cai thu vien (hf_xet tranh Xet 403)
get_ipython().system('pip install -q datasets tqdm huggingface_hub comet_ml python-dotenv scikit-learn pytest hf-transfer transformers hf_xet sentence-transformers')
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('Da cai thu vien, hf_transfer bat.')

In [ ]:
# 1.4. Nap .env + kiem tra secret
import os
from dotenv import load_dotenv
load_dotenv('.env') if os.path.exists('.env') else load_dotenv()
def _mask(t):
    return (t[:8] + '...' + t[-6:]) if t and len(t) > 16 else ('co' if t else 'TRONG')
for k in ['COMET_API_KEY', 'DISCORD_WEBHOOK_URL', 'HF_TOKEN']:
    print(' ', k, ':', _mask(os.environ.get(k, '')))

In [ ]:
# 1.5. (Tuy chon) override Discord webhook — de trong = dung .env/Kaggle Secret
DISCORD_WEBHOOK_OVERRIDE = ''   # <-- dan webhook vao day neu muon doi
import os
if DISCORD_WEBHOOK_OVERRIDE.strip():
    os.environ['DISCORD_WEBHOOK_URL'] = DISCORD_WEBHOOK_OVERRIDE.strip()
    print('Discord: override.')
else:
    print('Discord: dung .env / Kaggle Secret.')

## BƯỚC 2: CONFIG — CHỈ SỬA Ở ĐÂY (tuned)

In [ ]:
# ================== CONFIG (TUNED) ==================
EPOCHS    = 100
PATIENCE  = 15
RL_EPOCHS = 20
RL_LR     = 1e-5

LR      = 1e-3     # model nhe (CNN/ResNet/MLP proj)
LR_GPT2 = 1e-4     # decoder GPT-2 (pretrained lon) -> LR thap
WD      = 1e-4     # weight decay mac dinh
WD_GPT2 = 1e-2     # GPT-2 -> regularize manh hon

BATCH = {'light': 128, 'heavy': 64, 'gpt2': 32}

COMET_PROJECT = 'medvqa-generative-ablation-ep100-tuned'
HF_REPO       = 'VQA-DeepLearning/vqa-rad-generative-ep100-tuned'   # doi lai neu muon gop baseline

USE_HF_PUSH = True
USE_COMET   = True
USE_DISCORD = True
print('CONFIG(tuned):', EPOCHS, 'ep | pat', PATIENCE,
      '| lr', LR, '/ gpt2', LR_GPT2, '| wd', WD, '/ gpt2', WD_GPT2, '| repo', HF_REPO)

## BƯỚC 3: RUN MATRIX

In [ ]:
def _freeze_set(prefix, img, txt, batch, attn):
    base = {'img': img, 'txt': txt, 'dec': 'mlp', 'batch': batch, 'attn': attn}
    return [
        dict(base, name=prefix + '_all', freeze=None),
        dict(base, name=prefix + '_img', freeze='image'),
        dict(base, name=prefix + '_txt', freeze='text'),
        dict(base, name=prefix + '_dec', freeze='decoder'),
    ]

RUNS = {
    'A1': _freeze_set('A1_lstm', 'cnn', 'lstm', 'light', False)
        + _freeze_set('A1_trans', 'cnn', 'transformer', 'light', False),
    'A2': _freeze_set('A2_lstm', 'cnn', 'lstm', 'light', True)
        + _freeze_set('A2_trans', 'cnn', 'transformer', 'light', True),
    'P': [
        {'name': 'P_resnet_lstm',  'img': 'resnet18_frozen', 'txt': 'lstm',        'dec': 'mlp',  'batch': 'heavy', 'attn': True,  'freeze': None},
        {'name': 'P_resnet_trans', 'img': 'resnet18_frozen', 'txt': 'transformer', 'dec': 'mlp',  'batch': 'heavy', 'attn': True,  'freeze': None},
        {'name': 'P_pubmed_mlp',   'img': 'pubmedclip',      'txt': 'pubmedbert',  'dec': 'mlp',  'batch': 'heavy', 'attn': False, 'freeze': None},
        {'name': 'P_pubmed_gpt2',  'img': 'pubmedclip',      'txt': 'pubmedbert',  'dec': 'gpt2', 'batch': 'gpt2',  'attn': False, 'freeze': None},
    ],
}

SMOKE = [
    {'name': 'smoke_gpt2', 'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'batch': 'gpt2', 'attn': False, 'freeze': None, 'smoke': True},
]

R_RUN   = {'name': 'R_constrained', 'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'batch': 'gpt2', 'attn': False, 'freeze': None, 'ckpt': 'P_pubmed_gpt2', 'constrained': True}
D_TRAIN = {'name': 'D1_scst', 'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'batch': 'gpt2', 'attn': False, 'freeze': None, 'rl': True, 'load_checkpoint': 'runs/P_pubmed_gpt2_best.pt'}
D_EVAL  = {'name': 'D1_scst', 'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'batch': 'gpt2', 'attn': False, 'freeze': None, 'constrained': True}

print('Tong training run:', sum(len(v) for v in RUNS.values()), '| nhom:', list(RUNS))

## BƯỚC 4: HELPER TRAIN

In [ ]:
import subprocess

def _model(r):
    f = ['--image_encoder', r['img'], '--text_encoder', r['txt'], '--decoder', r['dec']]
    if r.get('attn'): f += ['--image_attention', '--text_attention']
    fr = r.get('freeze')
    if fr == 'image':   f += ['--freeze_image']
    if fr == 'text':    f += ['--freeze_text']
    if fr == 'decoder': f += ['--freeze_decoder']
    return f

def _common(push=True):
    f = ['--project_name', COMET_PROJECT]
    if USE_COMET:   f += ['--use_comet']
    if USE_DISCORD: f += ['--use_discord']
    if USE_HF_PUSH and push: f += ['--use_hf_push', '--hf_repo_id', HF_REPO]
    return f

def run_train(r):
    # 'python -u' = unbuffered -> output train.py HIEN THI LIVE (khong bi subprocess buffer,
    # tranh cam giac "dung hinh" khi dang tai model / train).
    cmd = ['python', '-u', 'train.py', '--run_name', r['name']] + _model(r)
    cmd += ['--batch_size', str(BATCH[r['batch']])]
    if r.get('smoke'):
        cmd += ['--smoke']
    else:
        is_gpt2 = (r['dec'] == 'gpt2')          # TINH CHINH: GPT-2 dung LR/WD rieng
        lr = LR_GPT2 if is_gpt2 else LR
        wd = WD_GPT2 if is_gpt2 else WD
        cmd += ['--epochs', str(EPOCHS), '--lr', str(lr), '--weight_decay', str(wd), '--patience', str(PATIENCE)]
    if r.get('rl'):
        cmd += ['--rl', '--rl_epochs', str(RL_EPOCHS), '--rl_lr', str(RL_LR), '--load_checkpoint', r['load_checkpoint']]
    cmd += _common(push=not r.get('smoke'))
    print('>>', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=False)

def train_group(g):
    for r in RUNS[g]:
        run_train(r)
    print('== Train xong nhom', g, '==')

## BƯỚC 5: Warm-cache + Smoke test
Cell dưới **tải trước** CLIP + PubMedBERT + GPT2 (~1.5GB, lần đầu — hiện progress ngay), rồi chạy smoke để kiểm tra pipeline. Các bước train sau tải lại từ cache nên nhanh, không bị cảm giác "đứng hình".

In [ ]:
# === WARM-CACHE (tai truoc de from_pretrained khong phai tai -> khong 429/treo) ===
# QUAN TRONG: KHONG bat bien offline nao (HF_HUB_OFFLINE/TRANSFORMERS_OFFLINE) vi chung
# CHAN CA PUSH HF. Da warm-cache thi from_pretrained online chi check nhe -> khong treo.
import os
for k in ['HF_HUB_OFFLINE', 'HF_DATASETS_OFFLINE', 'TRANSFORMERS_OFFLINE']:
    os.environ.pop(k, None)                      # bao dam KHONG offline -> push HF hoat dong
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'   # downloader chuan cho buoc warm-cache (on dinh)

from transformers import CLIPVisionModel, AutoModel, AutoTokenizer, GPT2LMHeadModel
from datasets import load_dataset
_BERT = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
print('1/4 CLIP...', flush=True);       CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch32')
print('2/4 PubMedBERT...', flush=True); AutoModel.from_pretrained(_BERT); AutoTokenizer.from_pretrained(_BERT)
print('3/4 GPT2...', flush=True);       GPT2LMHeadModel.from_pretrained('gpt2')
print('4/4 VQA-RAD...', flush=True);    load_dataset('flaviagiammarino/vqa-rad')
print('=> CACHE XONG. Online mode: model doc cache (khong treo), push HF hoat dong.', flush=True)

# Smoke test
for r in SMOKE:
    run_train(r)
print('Smoke xong — pipeline OK')

## BƯỚC 6: Train nhóm A1 (không attention)

In [ ]:
train_group('A1')

## BƯỚC 7: Train nhóm A2 (có attention)

In [ ]:
train_group('A2')

## BƯỚC 8: Train nhóm P (pretrained + generative)

In [ ]:
train_group('P')

## BƯỚC 9: Train nhóm D — RL / SCST
RL fine-tune trên `runs/P_pubmed_gpt2_best.pt` (đã có từ BƯỚC 8, cùng session).

In [ ]:
run_train(D_TRAIN)

## BƯỚC 10: EVAL ĐẦY ĐỦ (2-in-1)
Mỗi model load **1 lần** → test **cả VQA-RAD + RadImageNet** → push HF.
VQA-RAD → `<run>/metrics.json` · RadImageNet → `<run>/radimagenet/metrics.json` (lồng, không đè).

In [ ]:
import os, json, time, io, torch
from PIL import Image
from datasets import load_dataset
from torch.utils.data import DataLoader
from src.models.fusion import build_model
from src.train.engine import predict_answers
from src.train.metrics import score_predictions
from src.data.dataset import VQARADClsDataset, default_transform, build_text_tokenizer
from src.data.vqa_rad import normalize_answer
from src.integrations import hf_push

device   = 'cuda' if torch.cuda.is_available() else 'cpu'
PARQUET  = 'radimagenet_vqa_audit/radimagenet_vqa_500_test.parquet'
HF_TOKEN = os.environ.get('HF_TOKEN')

# vocab (train.py da luu san sau khi train)
with open('runs/vocab_answer2id.json') as f: answer2id = json.load(f)
with open('runs/vocab_q_vocab.json')  as f: q_vocab   = json.load(f)
id2answer = {i: a for a, i in answer2id.items()}
yes_id, no_id = answer2id.get('yes'), answer2id.get('no')

# tai 2 test set 1 lan
print('Tai VQA-RAD test...')
vqarad = [dict(x) for x in load_dataset('flaviagiammarino/vqa-rad')['test']]
print('Tai RadImageNet parquet...')
rad = [dict(x) for x in load_dataset('parquet', data_files=PARQUET, split='train')]
for _r in rad:
    im = _r.get('image')
    if isinstance(im, dict) and im.get('bytes'):
        _r['image'] = Image.open(io.BytesIO(im['bytes']))
TESTS = {'vqarad': (vqarad, 'runs'), 'radimagenet': (rad, 'runs_radimagenet')}

def _freeze(model, fr):
    if fr == 'image':
        for p in model.image.parameters(): p.requires_grad = False
    if fr == 'text':
        for p in model.text.parameters(): p.requires_grad = False
    if fr == 'decoder':
        for a in ['classifier','decoder','gpt2','gpt2_proj','init_decoder']:
            if hasattr(model, a):
                for p in getattr(model, a).parameters(): p.requires_grad = False

def eval_both(r, push=True):
    name, ckpt = r['name'], r.get('ckpt', r['name'])
    path = 'runs/' + ckpt + '_best.pt'
    if not os.path.exists(path):
        print('x khong co checkpoint:', path); return
    model = build_model(vocab_size=len(q_vocab), num_classes=len(answer2id),
                        image_encoder=r['img'], text_encoder=r['txt'], decoder=r['dec'],
                        image_attention=r.get('attn', False), text_attention=r.get('attn', False),
                        pretrained=False, max_len=32)
    model.load_state_dict(torch.load(path, map_location=device))
    _freeze(model, r.get('freeze'))
    model.to(device).eval()
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tok = build_text_tokenizer(r['txt'])
    tf  = default_transform(224, train=False,
                            imagenet=(r['img']=='resnet18_frozen'), clip=(r['img']=='pubmedclip'))
    print('>> [' + name + '] loaded — eval 2 test...')
    for tag, (recs, outdir) in TESTS.items():
        loader = DataLoader(VQARADClsDataset(recs, answer2id, q_vocab, tf, max_len=32,
                                             text_tokenizer=tok), batch_size=32, shuffle=False)
        t0 = time.time()
        hyps, p_yes = predict_answers(model, loader, id2answer, q_vocab, device,
                                      yes_id=yes_id, no_id=no_id,
                                      constrained_closed=r.get('constrained', False))
        m = score_predictions([normalize_answer(x['answer']) for x in recs], hyps, p_yes=p_yes)
        m['params_trainable'] = n_train
        m['inference_time_sec'] = round(time.time() - t0, 3)
        os.makedirs(outdir, exist_ok=True)
        fp = outdir + '/' + name + '_metrics.json'
        with open(fp, 'w') as f: json.dump(m, f, indent=2)
        sub = name if tag == 'vqarad' else name + '/radimagenet'
        if push and HF_TOKEN:
            hf_push.push_files(HF_REPO, [(fp, sub + '/metrics.json')], HF_TOKEN)
        print('   ', tag, '| em', round(m['em_overall'], 4), '| sem_open', m.get('sem_open'))
    del model; torch.cuda.empty_cache()

# EVAL TAT CA: moi run trong RUNS + R_constrained + D1_scst
EVAL_LIST = [rr for g in RUNS for rr in RUNS[g]] + [R_RUN, D_EVAL]
for r in EVAL_LIST:
    eval_both(r)
print('=== XONG EVAL 2 TEST ===')

## BƯỚC 11: Bảng tổng hợp + Backup

In [ ]:
# 2 bang tach biet (VQA-RAD in-domain vs RadImageNet cross-dataset)
get_ipython().system('python report.py --out-dir runs             --out-csv runs/summary_vqarad.csv')
get_ipython().system('python report.py --out-dir runs_radimagenet --out-csv runs_radimagenet/summary_radimagenet.csv')

In [ ]:
# Backup du phong: day toan bo runs/ trong 1 commit
from huggingface_hub import HfApi
import os
try:
    HfApi(token=os.environ['HF_TOKEN']).upload_folder(folder_path='runs', repo_id=HF_REPO, repo_type='model', allow_patterns=['*.json', '*.csv', '*.pt'])
    print('Backup runs/ trong 1 commit')
except Exception as e:
    print('Backup bo qua:', e)

---
## Phụ lục — CHỈ chạy khi MẤT SESSION
Kaggle không giữ `runs/` qua session mới. Nếu kernel restart giữa chừng, chạy 2 cell dưới để tải lại checkpoint đã push HF + tái tạo vocab, rồi tiếp tục từ bước còn dang dở.

In [ ]:
# (Phu luc) Tai lai TAT CA checkpoint da push HF ve runs/
from huggingface_hub import hf_hub_download
import os, shutil
os.makedirs('runs', exist_ok=True)
allnames = [r['name'] for g in RUNS for r in RUNS[g]] + ['D1_scst']
ok = 0
for name in allnames:
    try:
        p = hf_hub_download(repo_id=HF_REPO, filename=name + '/best.pt', token=os.environ.get('HF_TOKEN'))
        shutil.copy(p, 'runs/' + name + '_best.pt'); ok += 1
    except Exception:
        print('x chua co:', name)
print('Da tai', ok, '/', len(allnames), 'checkpoint')

In [ ]:
# (Phu luc) Tai tao vocab (deterministic seed 42, y het train.py) neu mat runs/vocab_*.json
import os, json
from datasets import load_dataset
from src.data.vqa_rad import build_answer_vocab, build_vocab, image_key, group_split
ds = load_dataset('flaviagiammarino/vqa-rad')
train_recs = [dict(r) for r in ds['train']]
keys = [image_key(r['image']) for r in train_recs]
tr_idx, _ = group_split(keys, val_ratio=0.1, seed=42)
tr = [train_recs[i] for i in tr_idx]
answer2id = build_answer_vocab([r['answer'] for r in tr])
q_vocab   = build_vocab([r['question'] for r in tr], [r['answer'] for r in tr])
os.makedirs('runs', exist_ok=True)
with open('runs/vocab_answer2id.json', 'w') as f: json.dump(answer2id, f, indent=2)
with open('runs/vocab_q_vocab.json',  'w') as f: json.dump(q_vocab,   f, indent=2)
print('Vocab: C =', len(answer2id), '| V =', len(q_vocab))